### Hi I am using this notebook to IPL Match Information Retrieval System.

### The goal is:

#### User asks a question → ChromaDB finds the most relevant match records → returns the top matching documents.

In [4]:
import pandas as pd
import numpy as np 
import ast


# =====================================================================
# DATA CLEANING LOGIC 
# =====================================================================

In [8]:
def clean_ipl_dataset(df):
    """
    Standardizes and cleans raw IPL match metrics, handling franchise 
    name changes, Super Over anomalies, and parsing squads.
    """

    cleaned_df = df.copy()

    # Standardize Franchise Names (including recent Bengaluru update)
    team_mapping = {
        'Delhi Daredevils': 'Delhi Capitals',
        'Kings XI Punjab': 'Punjab Kings',
        'Deccan Chargers': 'Sunrisers Hyderabad',
        'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
    }

    team_cols = ['team1', 'team2', 'toss_winner', 'winner']
    for col in team_cols:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].astype(str).str.strip()
            cleaned_df[col] = cleaned_df[col].replace(team_mapping)


    # Handle Super Over Flags & Margins
    if 'super_over' in cleaned_df.columns:
        cleaned_df['is_super_over'] = cleaned_df['super_over'].astype(str).str.upper().str.strip().isin(['Y', 'YES', '1', 'TRUE'])
        if 'win_by_runs' in cleaned_df.columns and 'win_by_wickets' in cleaned_df.columns:
            cleaned_df.loc[cleaned_df['is_super_over'], ['win_by_runs', 'win_by_wickets']] = 0
            
    # Normalize and Cross-Validate Margins
    if 'win_by_runs' in cleaned_df.columns and 'win_by_wickets' in cleaned_df.columns:
        cleaned_df['win_by_runs'] = pd.to_numeric(cleaned_df['win_by_runs'], errors='coerce').fillna(0).astype(int)
        cleaned_df['win_by_wickets'] = pd.to_numeric(cleaned_df['win_by_wickets'], errors='coerce').fillna(0).astype(int)
        
        cleaned_df['win_type'] = np.where(cleaned_df['win_by_runs'] > 0, 'Runs',
                                 np.where(cleaned_df['win_by_wickets'] > 0, 'Wickets', 
                                 np.where(cleaned_df.get('is_super_over', False), 'Super Over', 'No Result')))
        
        cleaned_df.loc[cleaned_df['win_type'] == 'Runs', 'win_by_wickets'] = 0
        cleaned_df.loc[cleaned_df['win_type'] == 'Wickets', 'win_by_runs'] = 0
    # Parse String Arrays into clean Python lists
    def clean_playing_xi(squad_data):
        if pd.isna(squad_data): return []
        if isinstance(squad_data, str):
            squad_data = squad_data.strip()
            if squad_data.startswith('[') and squad_data.endswith(']'):
                try: squad_data = ast.literal_eval(squad_data)
                except (ValueError, SyntaxError): squad_data = squad_data.replace('[', '').replace(']', '').split(',')
            else: squad_data = squad_data.split(',')
                
        if isinstance(squad_data, list):
            cleaned_squad = []
            for player in squad_data:
                p_clean = str(player).strip()
                for suffix in ['(c)', '(wk)', ' (c)', ' (wk)', '(sub)']:
                    p_clean = p_clean.replace(suffix, '')
                p_clean = p_clean.strip()
                if p_clean: cleaned_squad.append(p_clean)
            return cleaned_squad
        return []

    if 'playing_xi' in cleaned_df.columns:
        cleaned_df['playing_xi'] = cleaned_df['playing_xi'].apply(clean_playing_xi)
        
    return cleaned_df
        

# =====================================================================
# VECTOR DB SERIALIZATION LOGIC 
# =====================================================================

In [9]:
def serialize_row_for_chromadb(row):
    """
    Takes a single CLEANED row and returns a tuple: (narrative_text, metadata_dict)
    """
    # 1. Expand standard acronyms so the vector database captures searches for full names
    abbreviation_map = {
        'CSK': 'Chennai Super Kings', 'MI': 'Mumbai Indians', 'RCB': 'Royal Challengers Bengaluru',
        'KKR': 'Kolkata Knight Riders', 'SRH': 'Sunrisers Hyderabad', 'DC': 'Delhi Capitals',
        'PK': 'Punjab Kings', 'RR': 'Rajasthan Royals', 'GT': 'Gujarat Titans', 'LSG': 'Lucknow Super Giants'
    }
    
    def expand_team(name):
        return abbreviation_map.get(str(name).strip(), str(name).strip())

    season = row.get('season', 'Unknown Season')
    t1 = expand_team(row.get('team1', 'Team 1'))
    t2 = expand_team(row.get('team2', 'Team 2'))
    winner = expand_team(row.get('winner', 'No Result'))
    toss_winner = expand_team(row.get('toss_winner', 'Unknown'))
    
    toss_decision = str(row.get('toss_decision', '')).lower()
    venue = row.get('venue', 'an unknown stadium')
    pom = row.get('player_of_the_match', 'None')
    result = row.get('result', 'Match completed')
    
    # Clean handling of player arrays for text serialization
    squad = row.get('playing_xi', [])
    squad_text = f" The players in the squad included: {', '.join(squad)}." if squad else ""

    # Build the structural narrative string
    narrative_text = (
        f"In the {season} IPL season, a match was played between {t1} and {t2} at the {venue} stadium. "
        f"{toss_winner} won the toss and elected to {toss_decision} first. "
        f"The final outcome was: {result}. The official winner of the match was {winner}. "
        f"For a standout performance, {pom} was awarded the Player of the Match.{squad_text}"
    )

    # Build structural metadata dict for ChromaDB rigid filtering
    metadata = {
        "season": int(season) if str(season).isdigit() else 0,
        "team1": str(row.get('team1')),
        "team2": str(row.get('team2')),
        "winner": str(row.get('winner')),
        "player_of_the_match": str(pom)
    }

    return pd.Series([narrative_text, metadata])

# =====================================================================
# THE UNIFIED EXECUTION PIPELINE
# =====================================================================

In [10]:
if __name__ == "__main__":
    # Mocking your raw dirty data inputs
    raw_match_data = {
        'season': [2024],
        'team1': ['CSK'],
        'team2': ['MI'],
        'winner': ['CSK '], # Intentionally trailing space
        'toss_winner': ['MI'],
        'toss_decision': ['Bowl'],
        'venue': ['Chennai'],
        'player_of_the_match': ['Jadeja'],
        'result': ['CSK won by 6 wickets'],
        'win_by_runs': ["0"],     # Intentionally string type
        'win_by_wickets': [6],
        'super_over': ['N'],
        'playing_xi': ["['MS Dhoni (wk)', 'Jadeja (c)', 'Ruturaj Gaikwad']"]
    }
    
    df_raw = pd.DataFrame(raw_match_data)
    
    print("--- Executing Combined Pipeline ---")
    
    # Pass 1: Clear data defects out of the structural table
    df_cleaned = clean_ipl_dataset(df_raw)
    
    # Pass 2: Turn structural rows into specialized Vector Document outputs
    df_cleaned[['chroma_document', 'chroma_metadata']] = df_cleaned.apply(serialize_row_for_chromadb, axis=1)
    
    # Inspect final pipeline results ready for database insertion
    print("\n[PREPARED EMBEDDING DOCUMENT]:")
    print(df_cleaned['chroma_document'].iloc[0])
    
    print("\n[PREPARED METADATA DICTIONARY]:")
    print(df_cleaned['chroma_metadata'].iloc[0])

--- Executing Combined Pipeline ---

[PREPARED EMBEDDING DOCUMENT]:
In the 2024 IPL season, a match was played between Chennai Super Kings and Mumbai Indians at the Chennai stadium. Mumbai Indians won the toss and elected to bowl first. The final outcome was: CSK won by 6 wickets. The official winner of the match was Chennai Super Kings. For a standout performance, Jadeja was awarded the Player of the Match. The players in the squad included: MS Dhoni, Jadeja, Ruturaj Gaikwad.

[PREPARED METADATA DICTIONARY]:
{'season': 2024, 'team1': 'CSK', 'team2': 'MI', 'winner': 'CSK', 'player_of_the_match': 'Jadeja'}
